# 目次

- [1. 環境設定と入力探索](#sec-setup)
- [2. tomo 抽出](#sec-select)
- [3. 前処理（raw Image -> .npy）](#sec-preprocess)
- [4. CustomDataset と座標→ラベルマップ変換](#sec-dataset)
- [5. モデルと学習ループ](#sec-train)

この目次から各章へ移動できます。

<a href="https://www.kaggle.com/code/ugokorubeast/ugoko-byu?scriptVersionId=315947157" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## 1. 環境設定と入力探索
<a id="sec-setup"></a>

ライブラリ読み込み、seed 固定、入力ディレクトリの探索を行います。

In [2]:
from pathlib import Path
import random
import hashlib

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ===== Phase 1 minimal training config =====
SEED = 42
NUM_TOMOS = 2
DEPTH = 16
IMG_SIZE = 96
BATCH_SIZE = 1
NUM_WORKERS = 0
EPOCHS = 1
LR = 1e-3
LABEL_RADIUS = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ../input, ./input, train_data などから tomo_* ディレクトリがある場所を自動検出
candidate_roots = [
    Path('../input'),
    Path('../input/competitions/byu-locating-bacterial-flagellar-motors-2025/train'),
    Path('./input'),
    Path('train_data'),
    Path('../train_data'),
]

input_root = None
for root in candidate_roots:
    if root.exists() and any(p.is_dir() and p.name.startswith('tomo_') for p in root.iterdir()):
        input_root = root
        break

if input_root is None:
    raise FileNotFoundError(
        "tomo_* ディレクトリが見つかりません。候補: ../input, ./input, train_data, ../train_data"
    )

print(f"[INFO] input_root = {input_root.resolve()}")

[INFO] input_root = /Users/yuhaogao/workspace/BYU-competition/train_data


## 2. tomo 抽出
<a id="sec-select"></a>

利用可能な tomo から seed 固定で学習対象を抽出します。

In [ ]:
all_tomos = sorted([
    p for p in input_root.iterdir()
    if p.is_dir() and p.name.startswith('tomo_')
])

if len(all_tomos) < NUM_TOMOS:
    raise ValueError(f"tomo フォルダ数が不足しています: found={len(all_tomos)}, required={NUM_TOMOS}")

# seed 固定で 2 件ランダム抽出
selected_tomos = random.sample(all_tomos, k=NUM_TOMOS)

print('[INFO] selected_tomos:')
for p in selected_tomos:
    print(' -', p.name)

[INFO] selected_tomos:
 - tomo_2c607f
 - tomo_2bb588


## 3. 前処理（raw Image -> .npy）
<a id="sec-preprocess"></a>

生データのスライス画像を読み込み、学習で使いやすい `.npy` ボリュームとして保存します。

In [ ]:
processed_root = Path('./processed_tomos')
processed_root.mkdir(parents=True, exist_ok=True)


def load_and_resize_volume(tomo_dir: Path, img_size: int):
    slice_paths = sorted(tomo_dir.glob('slice_*.jpg'))
    if not slice_paths:
        slice_paths = sorted(tomo_dir.glob('*.jpg'))
    if not slice_paths:
        raise FileNotFoundError(f'画像スライスが見つかりません: {tomo_dir}')

    volume = []
    for path in slice_paths:
        image = Image.open(path).convert('L').resize((img_size, img_size))
        array = np.asarray(image, dtype=np.float32) / 255.0
        volume.append(array)

    return np.stack(volume, axis=0)  # [D, H, W]


# 2-1: 生データ -> .npy 前処理
# NOTE: まずは選択した tomo のみを保存。全件処理するなら preprocess_targets = all_tomos に変更。
preprocess_targets = selected_tomos

for tomo_dir in preprocess_targets:
    volume = load_and_resize_volume(tomo_dir, IMG_SIZE)
    out_path = processed_root / f'{tomo_dir.name}.npy'
    np.save(out_path, volume.astype(np.float32))
    print(f'[PREPROCESS] {tomo_dir.name} -> {out_path} shape={volume.shape}')

print(f'[INFO] processed_root = {processed_root.resolve()}')

[PREPROCESS] tomo_2c607f -> processed_tomos/tomo_2c607f.npy shape=(300, 96, 96)
[PREPROCESS] tomo_2bb588 -> processed_tomos/tomo_2bb588.npy shape=(300, 96, 96)
[INFO] processed_root = /Users/yuhaogao/workspace/BYU-competition/processed_tomos


## 4. CustomDataset と座標→ラベルマップ変換
<a id="sec-dataset"></a>

CSV 座標を読み取り、3D ラベルマップへ変換して input/target を返す Dataset を作成し、`get_dataset` / `get_dataloader` で DataLoader を組み立てます。

In [ ]:
class CustomDataset(Dataset):
    """2-2 実装: 座標アノテーションを 3D ラベルマップへ変換して返す。"""

    def __init__(self, tomo_dirs, depth=16, img_size=96, label_radius=1, processed_root: Path | None = None):
        self.tomo_dirs = list(tomo_dirs)
        self.depth = depth
        self.img_size = img_size
        self.label_radius = label_radius
        self.processed_root = processed_root
        self.coord_meta = self._load_coord_metadata()

    def __len__(self):
        return len(self.tomo_dirs)

    def _load_slice(self, path: Path):
        img = Image.open(path).convert('L').resize((self.img_size, self.img_size))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return arr

    def _load_coord_metadata(self):
        csv_candidates = [Path('./folds_all.csv'), Path('./data/processed/folds_all.csv')]
        csv_path = next((p for p in csv_candidates if p.exists()), None)
        if csv_path is None:
            print('[WARN] folds_all.csv が見つからないため、ラベルは全て 0 になります。')
            return {}

        df = pd.read_csv(csv_path)
        grouped = {}
        for tomo_id, g in df.groupby('tomo_id'):
            first = g.iloc[0]
            coords = []
            # 列名に空白があるため、属性アクセスではなく列参照で座標を取得する
            for z, y, x in g[['Motor axis 0', 'Motor axis 1', 'Motor axis 2']].to_numpy(dtype=float):
                if z >= 0 and y >= 0 and x >= 0:
                    coords.append((z, y, x))

            grouped[tomo_id] = {
                'orig_shape': (
                    int(first['Array shape (axis 0)']),
                    int(first['Array shape (axis 1)']),
                    int(first['Array shape (axis 2)']),
                ),
                'coords': coords,
            }
        return grouped

    def _scale_coord(self, coord, src_len, dst_len):
        if src_len <= 1:
            return 0
        return int(round(coord * (dst_len - 1) / (src_len - 1)))

    def _load_volume_from_images(self, tomo_dir: Path):
        slices = sorted(tomo_dir.glob('slice_*.jpg'))
        if not slices:
            slices = sorted(tomo_dir.glob('*.jpg'))
        if not slices:
            raise FileNotFoundError(f'画像スライスが見つかりません: {tomo_dir}')

        # z 方向は全体から等間隔サンプリングして depth に揃える
        if len(slices) >= self.depth:
            sample_idx = np.linspace(0, len(slices) - 1, self.depth).astype(int)
            selected = [slices[i] for i in sample_idx]
        else:
            selected = slices + [slices[-1]] * (self.depth - len(slices))

        return np.stack([self._load_slice(p) for p in selected], axis=0)  # [D, H, W]

    def _match_depth(self, vol):
        d = vol.shape[0]
        if d == self.depth:
            return vol
        if d > self.depth:
            idx = np.linspace(0, d - 1, self.depth).astype(int)
            return vol[idx]
        pad = np.repeat(vol[-1:, :, :], self.depth - d, axis=0)
        return np.concatenate([vol, pad], axis=0)

    def _load_volume(self, tomo_dir: Path):
        # 2-1 の出力を優先して読み込み、ない場合のみ raw 画像から作る
        if self.processed_root is not None:
            npy_path = self.processed_root / f'{tomo_dir.name}.npy'
            if npy_path.exists():
                vol = np.load(npy_path).astype(np.float32)
                return self._match_depth(vol)

        return self._load_volume_from_images(tomo_dir)

    def _make_label_map(self, tomo_id: str):
        label = np.zeros((self.depth, self.img_size, self.img_size), dtype=np.float32)
        meta = self.coord_meta.get(tomo_id)
        if meta is None or len(meta['coords']) == 0:
            return label

        src_d, src_h, src_w = meta['orig_shape']
        r = self.label_radius
        for z0, y0, x0 in meta['coords']:
            z = self._scale_coord(z0, src_d, self.depth)
            y = self._scale_coord(y0, src_h, self.img_size)
            x = self._scale_coord(x0, src_w, self.img_size)

            z_min, z_max = max(0, z - r), min(self.depth, z + r + 1)
            y_min, y_max = max(0, y - r), min(self.img_size, y + r + 1)
            x_min, x_max = max(0, x - r), min(self.img_size, x + r + 1)
            label[z_min:z_max, y_min:y_max, x_min:x_max] = 1.0

        return label

    def __getitem__(self, idx):
        tomo_dir = self.tomo_dirs[idx]
        vol = self._load_volume(tomo_dir)
        label = self._make_label_map(tomo_dir.name)

        x = torch.from_numpy(vol).unsqueeze(0)      # [1, D, H, W]
        target = torch.from_numpy(label).unsqueeze(0)  # [1, D, H, W]
        return {'input': x, 'target': target, 'tomo_id': tomo_dir.name}


# 2-3: get_dataset / get_dataloader 実装
# mode ごとの既定値を分けつつ、shuffle/batch/num_workers を明示的に扱う
DATALOADER_CFG = {
    'train': {
        'batch_size': BATCH_SIZE,
        'shuffle': True,
        'num_workers': NUM_WORKERS,
    },
    'val': {
        'batch_size': BATCH_SIZE,
        'shuffle': False,
        'num_workers': NUM_WORKERS,
    },
}


def get_dataset(tomo_dirs, mode='train'):
    return CustomDataset(
        tomo_dirs,
        depth=DEPTH,
        img_size=IMG_SIZE,
        label_radius=LABEL_RADIUS,
        processed_root=processed_root,
    )


def get_dataloader(dataset, mode='train', shuffle=None, batch_size=None, num_workers=None):
    cfg = DATALOADER_CFG.get(mode, DATALOADER_CFG['train'])

    if shuffle is None:
        shuffle = cfg['shuffle']
    if batch_size is None:
        batch_size = cfg['batch_size']
    if num_workers is None:
        num_workers = cfg['num_workers']

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
    )


train_ds = get_dataset(selected_tomos, mode='train')
train_loader = get_dataloader(train_ds, mode='train')

sample = next(iter(train_loader))
print('[INFO] train_loader cfg =', DATALOADER_CFG['train'])
print('[INFO] sample input shape =', tuple(sample['input'].shape))
print('[INFO] sample target shape =', tuple(sample['target'].shape))
print('[INFO] sample target positive voxels =', int(sample['target'].sum().item()))

[INFO] sample input shape = (1, 1, 16, 96, 96)
[INFO] sample target shape = (1, 1, 16, 96, 96)
[INFO] sample target positive voxels = 27


## 5. モデルと学習ループ
<a id="sec-train"></a>

3D CNN を学習し、ラベルマップに対する損失で更新します。

In [7]:
class Tiny3DAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(16, 8, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(8, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Tiny3DAutoEncoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

print(f'[INFO] device = {device}')
print(f'[INFO] steps_per_epoch = {len(train_loader)}')

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader, start=1):
        x = batch['input'].to(device)
        target = batch['target'].to(device)

        optimizer.zero_grad(set_to_none=True)
        y = model(x)
        loss = criterion(y, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        current_lr = optimizer.param_groups[0]['lr']
        print(f'[train] epoch={epoch} step={step}/{len(train_loader)} loss={loss.item():.6f} lr={current_lr:.2e}')

    epoch_loss = running_loss / max(1, len(train_loader))
    print(f'[epoch] epoch={epoch} avg_loss={epoch_loss:.6f}')

[INFO] device = cpu
[INFO] steps_per_epoch = 2
[train] epoch=1 step=1/2 loss=0.711567 lr=1.00e-03
[train] epoch=1 step=2/2 loss=0.707990 lr=1.00e-03
[epoch] epoch=1 avg_loss=0.709779
